In [7]:
#IMPORTS 
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import shap

from scipy import stats
from statsmodels.stats.multitest import multipletests

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

In [8]:
#CONFIGURATION

DATA_PATH = "alzheimers_disease_data.csv"

# Second dataset is optional for now.
# Fill this in later once the second dataset is chosen.
DATA_PATH_2 = None
SECOND_DATASET_NAME = None  # Example: "OASIS-2" or "Global Alzheimer Kaggle"

TARGET = "Diagnosis"
RANDOM_STATE = 42

CV_FOLDS = 5
N_BOOT = 1000
N_PERM = 200

OUTPUT_DIR = Path("results")
OUTPUT_DIR.mkdir(exist_ok=True)

FAIRNESS_COLS = ["Gender", "AgeGroup"]

In [10]:
#FEATURE SETS

FEATURE_SETS = {
    "FS1_lifestyle": [
        "Age", "Gender", "Ethnicity", "EducationLevel", "BMI",
        "Smoking", "AlcoholConsumption", "PhysicalActivity",
        "DietQuality", "SleepQuality",
    ],

    "FS2_lifestyle_medical": [
        "Age", "Gender", "Ethnicity", "EducationLevel", "BMI",
        "Smoking", "AlcoholConsumption", "PhysicalActivity",
        "DietQuality", "SleepQuality",
        "FamilyHistoryAlzheimers", "CardiovascularDisease", "Diabetes",
        "Depression", "HeadInjury", "Hypertension",
        "SystolicBP", "DiastolicBP",
        "CholesterolTotal", "CholesterolLDL",
        "CholesterolHDL", "CholesterolTriglycerides",
    ],

    "FS3_full": [
        "Age", "Gender", "Ethnicity", "EducationLevel", "BMI",
        "Smoking", "AlcoholConsumption", "PhysicalActivity",
        "DietQuality", "SleepQuality",
        "FamilyHistoryAlzheimers", "CardiovascularDisease", "Diabetes",
        "Depression", "HeadInjury", "Hypertension",
        "SystolicBP", "DiastolicBP",
        "CholesterolTotal", "CholesterolLDL",
        "CholesterolHDL", "CholesterolTriglycerides",
        "MemoryComplaints", "Confusion", "BehavioralProblems",
        "ADL", "Disorientation", "PersonalityChanges",
        "DifficultyCompletingTasks", "Forgetfulness",
    ],
}

In [11]:
#LOAD PRIMARY DATASET

def load_primary_data(path: str) -> pd.DataFrame:
    """
    Load the primary Alzheimer dataset.

    - Removes identifier/confidential columns.
    - Creates AgeGroup for subgroup/fairness analysis only.
    - Does not scale variables here. Scaling is handled inside model pipelines.
    """

    df = pd.read_csv(path)

    df = df.drop(columns=["PatientID", "DoctorInCharge"], errors="ignore")

    if "Age" in df.columns:
        df["AgeGroup"] = pd.cut(
            df["Age"],
            bins=[0, 59, 69, 79, 120],
            labels=["<60", "60-69", "70-79", "80+"],
            right=True
        )

    return df


def inspect_class_balance(y: pd.Series) -> dict:
    """
    Print and return class distribution.
    """

    counts = y.value_counts().sort_index()
    ratio = counts.min() / counts.max()

    print("\nClass balance")
    print("-" * 40)

    for cls, n in counts.items():
        print(f"Class {cls}: {n} ({100 * n / len(y):.1f}%)")

    print(f"Minority / majority ratio: {ratio:.3f}")

    return {
        "counts": counts.to_dict(),
        "ratio": round(float(ratio), 4),
    }

In [13]:
#TRAIN TEST SPLIT 

def make_splits(df: pd.DataFrame, features: list):
    """
    Create stratified 70/15/15 train, validation, and test splits.
    """

    model_features = [f for f in features if f not in ["AgeGroup", TARGET]]

    X = df[model_features].copy()
    y = df[TARGET].copy()

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X,
        y,
        test_size=0.15,
        stratify=y,
        random_state=RANDOM_STATE,
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval,
        y_trainval,
        test_size=0.15 / 0.85,
        stratify=y_trainval,
        random_state=RANDOM_STATE,
    )

    return X_train, X_val, X_test, y_train, y_val, y_test